In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
import numpy as np
import os
import io

ModuleNotFoundError: No module named 'torchvision'

In [2]:
!git clone https://github.com/aaryankdk/aslive

Cloning into 'aslive'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 27 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 8.12 MiB | 13.24 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [3]:
VIDEO_DIR = "/content/aaryankdk/aslive"


In [4]:

class MediaPipeDataset(Dataset):
    def __init__(self, x_path, y_path, augment=False):
        X = np.load(x_path, allow_pickle=True)
        Y = np.load(y_path, allow_pickle=True)

        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.long)
        self.augment = augment

        if self.X.shape[1] == 30 and self.X.shape[2] == 147:
            self.X = self.X.permute(0, 2, 1)

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        x, y = self.X[idx], self.Y[idx]

        # If training, add slight geometric noise to simulate different coordinates
        if self.augment:
            noise = torch.randn_like(x) * 0.005 # Tiny standard deviation
            x = x + noise

        return x, y
# Training Split
X_TRAIN_PATH = "/content/aslive/preprocessed/train/X_train.npy"
Y_TRAIN_PATH = "/content/aslive/preprocessed/train/y_train.npy" # Fixed to lowercase y

# Validation Split
X_VAL_PATH = "/content/aslive/preprocessed/val/X_val.npy"
Y_VAL_PATH = "/content/aslive/preprocessed/val/y_val.npy" # Fixed to lowercase y

# Testing Split
X_TEST_PATH = "/content/aslive/preprocessed/test/X_test.npy"
Y_TEST_PATH = "/content/aslive/preprocessed/test/y_test.npy" # Fixed to lowercase y

# Initialize your train dataset now
train_dataset = MediaPipeDataset(X_TRAIN_PATH, Y_TRAIN_PATH)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset  = MediaPipeDataset(X_VAL_PATH, Y_VAL_PATH)
val_loader  = DataLoader(val_dataset, batch_size=32, shuffle=False)

test_dataset = MediaPipeDataset(X_TEST_PATH, Y_TEST_PATH)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [5]:
class Spatio_Temporal_1D_CNN(nn.Module):
    def __init__(self, num_classes=100):
        super(Spatio_Temporal_1D_CNN, self).__init__()


        self.spatial_conv = nn.Conv1d(in_channels=147, out_channels=64, kernel_size=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu1 = nn.ReLU()

        self.temporal_conv = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu2 = nn.ReLU()

        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten()

        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(in_features=128, out_features=num_classes)

    def forward(self, x):
        x = self.spatial_conv(x)
        x = self.bn1(x)
        x = self.relu1(x)

        x = self.temporal_conv(x)
        x = self.bn2(x)
        x = self.relu2(x)

        x = self.global_avg_pool(x)
        x = self.flatten(x)
        x = self.dropout(x) # Applied only during training
        x = self.fc(x)
        return x

In [6]:
#import torch
#import torch.nn as nn

#class Spatio_Temporal_1D_CNN(nn.Module):
 #   def __init__(self, num_classes=100):
  #      super(Spatio_Temporal_1D_CNN, self).__init__()
#
 #       #===========Spatial-Embedding-Domain=====================================================
  #      self.spatial_conv = nn.Conv1d(in_channels=147, out_channels=64, kernel_size=1)
   #     self.relu1 = nn.ReLU()

        #===========Temporal-Modeling-Domain=====================================================
    #    self.temporal_conv = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
     #   self.relu2 = nn.ReLU()

        #===========Dimensionality-Reduction=====================================================
      #  self.global_avg_pool = nn.AdaptiveAvgPool1d(1)

        #===========100-Word-Classification-Step==================================================
       # self.flatten = nn.Flatten()
        #self.fc = nn.Linear(in_features=128, out_features=num_classes)
        #self.softmax = nn.Softmax(dim=1)

    #===========forward=====================================================================
    #def forward(self, x):
     #   x = self.spatial_conv(x)
      #  x = self.relu1(x)

       # x = self.temporal_conv(x)
        #x = self.relu2(x)

        #x = self.global_avg_pool(x)
        #x = self.flatten(x)

        #x = self.fc(x)
        #x = self.softmax(x)

        #return x


In [7]:

print("Files in train folder:", os.listdir("/content/aslive/preprocessed/train"))
print("Files in val folder:  ", os.listdir("/content/aslive/preprocessed/val"))
print("Files in test folder: ", os.listdir("/content/aslive/preprocessed/test"))

Files in train folder: ['X_train.npy', 'y_train.npy']
Files in val folder:   ['y_val.npy', 'X_val.npy']
Files in test folder:  ['y_test.npy', 'X_test.npy']


In [8]:
NUM_CLASSES = 100
BATCH_SIZE = 32
LEARNING_RATE = 0.001
NUM_EPOCHS = 200

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = Spatio_Temporal_1D_CNN(num_classes=NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Starting Training...")
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        correct_predictions += (predicted == labels).sum().item()
        total_samples += labels.size(0)

    # Calculate average epoch loss and accuracy
    epoch_loss = running_loss / total_samples
    epoch_acc = (correct_predictions / total_samples) * 100

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

print("Training Complete!")

Using device: cuda
Starting Training...
Epoch [1/200] | Loss: 4.6481 | Accuracy: 1.60%
Epoch [2/200] | Loss: 4.3371 | Accuracy: 6.15%
Epoch [3/200] | Loss: 4.1471 | Accuracy: 9.49%
Epoch [4/200] | Loss: 3.9677 | Accuracy: 10.29%
Epoch [5/200] | Loss: 3.7785 | Accuracy: 16.58%
Epoch [6/200] | Loss: 3.6523 | Accuracy: 16.44%
Epoch [7/200] | Loss: 3.5183 | Accuracy: 21.12%
Epoch [8/200] | Loss: 3.3910 | Accuracy: 25.67%
Epoch [9/200] | Loss: 3.2384 | Accuracy: 27.41%
Epoch [10/200] | Loss: 3.1273 | Accuracy: 30.48%
Epoch [11/200] | Loss: 2.9952 | Accuracy: 35.43%
Epoch [12/200] | Loss: 2.8835 | Accuracy: 33.82%
Epoch [13/200] | Loss: 2.7372 | Accuracy: 41.18%
Epoch [14/200] | Loss: 2.6947 | Accuracy: 37.70%
Epoch [15/200] | Loss: 2.5695 | Accuracy: 42.65%
Epoch [16/200] | Loss: 2.4773 | Accuracy: 46.12%
Epoch [17/200] | Loss: 2.3718 | Accuracy: 46.93%
Epoch [18/200] | Loss: 2.2789 | Accuracy: 47.73%
Epoch [19/200] | Loss: 2.2490 | Accuracy: 49.33%
Epoch [20/200] | Loss: 2.1494 | Accuracy:

In [9]:
def evaluate_model(loader, dataset_name="Validation"):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    final_loss = running_loss / total
    final_acc = (correct / total) * 100
    print(f"=== {dataset_name} Results ===")
    print(f"Loss: {final_loss:.4f}")
    print(f"Accuracy: {final_acc:.2f}%\n")
    return final_acc

# Run the evaluation
val_acc = evaluate_model(val_loader, "Validation")
test_acc = evaluate_model(test_loader, "Testing")

=== Validation Results ===
Loss: 1.6890
Accuracy: 62.42%

=== Testing Results ===
Loss: 2.0330
Accuracy: 64.00%



In [10]:
torch.save(model, "complete_spatio_temporal_model.pth")

In [10]:
!pip install ai-edge-torch-nightly torchao tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 523.6/523.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.5/623.5 MB 846.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: h5py
    Found exist

In [11]:
!pip install -q -U torch torchvision litert-torch numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/

In [13]:
!pip install litert-torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.8/575.8 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.3/419.3 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 20.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_e

In [14]:
import litert_torch
import torch
print(litert_torch.__version__ if hasattr(litert_torch, "__version__") else "imported OK")
print(torch.__version__)

0.9.1
2.11.0+cu128


In [15]:
from google.colab import files
uploaded = files.upload()
pth_path = list(uploaded.keys())[0]
print(f"Uploaded: {pth_path}")

Saving complete_spatio_temporal_model.pth to complete_spatio_temporal_model (1).pth
Uploaded: complete_spatio_temporal_model (1).pth


In [16]:
import torch

try:
    model = torch.load(pth_path, map_location="cpu", weights_only=False)
    model.eval()
    print("Loaded a full model object directly.")
except Exception as e:
    print(f"Direct load failed ({e}) — falling back to state_dict loading below.")

    import torchvision
    model = torchvision.models.complete_spatio_temporal_model()  # <-- REPLACE with your architecture
    state_dict = torch.load(pth_path, map_location="cpu", weights_only=True)
    model.load_state_dict(state_dict)
    model.eval()
    print("Loaded weights into model architecture.")

Loaded a full model object directly.


In [17]:
import litert_torch

T = 30  # <-- replace with your actual sequence length
sample_inputs = (torch.randn(1, 147, T),)

edge_model = litert_torch.convert(model, sample_inputs)
edge_model.export("converted_model.tflite")
print("Saved converted_model.tflite")

(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:01) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


(00:01) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:00)

...

(00:03) [ DONE] LiteRT-Torch Convert > Run LiteRT Converter Passes (+00:00)

(00:03) [ DONE] LiteRT-Torch Convert (+00:03)

(00:00) [START] Write Model to converted_model.tflite

(00:00) [ DONE] Write Model to converted_model.tflite (+00:00)

Saved converted_model.tflite


In [18]:
from google.colab import files
files.download("converted_model.tflite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
!pip install torchinfo

In [21]:
from torchinfo import summary

In [21]:
model = Spatio_Temporal_1D_CNN(num_classes=100)
summary(model, input_size=(1, 147, 30))

Layer (type:depth-idx)                   Output Shape              Param #
Spatio_Temporal_1D_CNN                   [1, 100]                  --
├─Conv1d: 1-1                            [1, 64, 30]               9,472
├─BatchNorm1d: 1-2                       [1, 64, 30]               128
├─ReLU: 1-3                              [1, 64, 30]               --
├─Conv1d: 1-4                            [1, 128, 30]              24,704
├─BatchNorm1d: 1-5                       [1, 128, 30]              256
├─ReLU: 1-6                              [1, 128, 30]              --
├─AdaptiveAvgPool1d: 1-7                 [1, 128, 1]               --
├─Flatten: 1-8                           [1, 128]                  --
├─Dropout: 1-9                           [1, 128]                  --
├─Linear: 1-10                           [1, 100]                  12,900
Total params: 47,460
Trainable params: 47,460
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 1.04
Input size (MB): 0.02
Forward/ba